# Stochastic Calculus from Zero · Episode 9: Brownian motion, defined

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path and every world in this notebook is simulated or made up, with fixed random seeds so the numbers repeat.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. Episode 1's path · 4. Why the walk converges · 5. Covariance · 6. Scaling · 7. This episode's question · 8. Exercises · 9. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 9,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)

## 2. Last episode's answer

If $Z = 1$ in every world then $Q = P$: the real-world odds already make the discounted share a martingale, which happens when $p = q = 0.55$.

In [ ]:
out["prev"] = {"p_equals_q": 0.55}

## 3. Episode 1's path

**Brownian motion** $W$ is pinned down by four properties:

1. $W_0 = 0$;
2. **independent increments**: $W_t - W_s$ is independent of everything up to time $s$;
3. **normal increments**: $W_t - W_s$ is normal with mean 0 and variance $t - s$;
4. **continuous paths**.

We simulate it on a grid by adding independent normal steps with variance $\Delta t$, which gets properties 1-3 exactly at the grid points. The first path seed 2026 draws is Episode 1's path.

In [ ]:
rng = np.random.default_rng(seed)
t, W, dW = brownian_paths(1, 1_000, 1.0, rng)
assert abs(W[0, -1] - 0.666204) < 1e-6          # the same path as Episode 1
every = 2
out["path"] = {"t": t[::every].round(4).tolist(), "path": [W[0, ::every].round(4).tolist()], "W_T": float(W[0, -1]),
               "steps": 1_000,
               "ticks": [{"x": x, "label": lab} for x, lab in ((0, "0"), (0.25, "3m"), (0.5, "6m"), (0.75, "9m"), (1, "1y"))]}

## 4. Why the walk converges

A fair $\pm 1$ walk with $n$ steps, scaled by $1/\sqrt{n}$, ends at time 1 at $M_n/\sqrt{n}$. Its exact distribution is binomial. Compare its distribution function with the normal one, $N(x)$, and take the largest gap (the **Kolmogorov distance**). It shrinks like $1/\sqrt{n}$: the central limit theorem at time 1. **Donsker's theorem** says more: the whole scaled path converges to Brownian motion, not just its end point.

In [ ]:
norm_cdf = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))
ks = []
for n in (10, 100, 1_000, 10_000):
    k = np.arange(n + 1)
    pmf = np.exp([math.lgamma(n + 1) - math.lgamma(i + 1) - math.lgamma(n - i + 1) - n * math.log(2) for i in k])
    x = (2 * k - n) / math.sqrt(n)
    Nx = np.array([norm_cdf(v) for v in x])
    cdf = np.cumsum(pmf)
    gap = max(np.abs(cdf - Nx).max(), np.abs(np.concatenate([[0], cdf[:-1]]) - Nx).max())   # both sides of each jump
    ks.append({"steps": n, "gap": float(gap)})
assert all(a["gap"] > b["gap"] for a, b in zip(ks, ks[1:]))
out["ks"] = {"rows": ks, "g10": ks[0]["gap"], "g10000": ks[-1]["gap"]}
pd.DataFrame(ks)

## 5. Covariance

For $s \le t$, $W_t = W_s + (W_t - W_s)$ and the increment is independent of $W_s$, so $\mathrm{Cov}(W_s, W_t) = \mathrm{Var}(W_s) = s$. In general $\mathrm{Cov}(W_s, W_t) = \min(s, t)$.

In [ ]:
_, Wc, _ = brownian_paths(20_000, 100, 1.0, rng)
pairs = []
for s_, t_ in ((0.25, 0.75), (0.5, 0.5), (0.5, 1.0), (0.75, 1.0)):
    i, j = round(s_ * 100), round(t_ * 100)
    pairs.append({"s": s_, "t": t_, "cov": float(np.mean(Wc[:, i] * Wc[:, j])), "theory": min(s_, t_)})
    assert abs(pairs[-1]["cov"] - min(s_, t_)) < 0.03
inc_corr = float(np.corrcoef(Wc[:, 25], Wc[:, 75] - Wc[:, 25])[0, 1])
assert abs(inc_corr) < 2 / math.sqrt(20_000)     # within two standard errors of zero: "essentially zero"
out["cov"] = {"rows": pairs, "c_25_75": pairs[0]["cov"], "inc_corr": inc_corr}
pd.DataFrame(pairs)

## 6. Scaling

Speed time up by $c$ and shrink heights by $\sqrt{c}$: $\{W_{ct}/\sqrt{c}\}$ is again a Brownian motion. Check: at $c = 1/4$, $W_{0.25}/\sqrt{0.25}$ has variance 1, the same as $W_1$.

In [ ]:
scaled = Wc[:, 25] / math.sqrt(0.25)
out["scaling"] = {"c": 0.25, "var_scaled": float(scaled.var()), "var_W1": float(Wc[:, -1].var())}
assert abs(out["scaling"]["var_scaled"] - 1) < 0.04
out["scaling"]

## 7. This episode's question

$\mathrm{Cov}(W_2, W_5) = \min(2, 5) = 2$.

In [ ]:
out["question"] = {"s": 2, "t": 5, "answer": 2}

## 8. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** What is $\mathrm{Var}(W_3 - W_1)$?

<details><summary>Answer</summary>

The increment over $[1, 3]$ is normal with variance $3 - 1 = 2$.

</details>

**Exercise 2 (By hand).** What is $\mathrm{Cov}(W_1, W_4 - W_1)$?

<details><summary>Answer</summary>

0: the increment after time 1 is independent of $W_1$.

</details>

**Exercise 3 (By hand).** Show that $\mathrm{Cov}(W_s, W_t) = s$ for $s < t$ using only properties 2 and 3.

<details><summary>Answer</summary>

$\mathrm{Cov}(W_s, W_t) = \mathrm{Cov}(W_s, W_s) + \mathrm{Cov}(W_s, W_t - W_s) = \mathrm{Var}(W_s) + 0 = s$.

</details>

**Exercise 4 (In code).** Plot the Kolmogorov distance of Section 4 against the number of steps on log-log axes. What is the slope?

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
x = np.log([r["steps"] for r in ks]); y = np.log([r["gap"] for r in ks])
slope = np.polyfit(x, y, 1)[0]
assert -0.6 < slope < -0.4
print(f"slope {slope:.3f}: the gap shrinks like 1/sqrt(n)")

**Exercise 5 (In code).** Check the scaling property for $c = 1/2$: what is the variance of $W_{0.5}/\sqrt{0.5}$?

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
v = float((Wc[:, 50] / math.sqrt(0.5)).var())
assert abs(v - 1) < 0.04
print(f"variance of W_0.5 / sqrt(0.5): {v:.3f}")

## 9. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")